# L22 · 导数（derivative） — 切线（tangent line）斜率（tangent slope）、极限定义（limit definition）、数值微分（numerical differentiation） vs 解析微分（analytical differentiation）

**目标**：导数就是「输入动一点，输出动多少」——曲线的斜率。

**为什么对 Aurora 重要**：`numeric_derivative` 是验证反向传播（backpropagation，BP）正确性的标准工具——用数值梯度（numerical gradient）和解析梯度（analytical gradient）逐元素比对，确认 MFCC（Mel-Frequency Cepstral Coefficients，MFCC）、梅尔滤波器（mel filterbank）等模块的求导实现没有错误。

← **上一课**　[L21 · 矩阵即滤波](../2_linear_algebra/L21_aurora_matrices.ipynb)

> 上节课学习了 **矩阵即滤波**：DFT 矩阵 / Mel 矩阵：音频处理 = 矩阵乘法。  
> 本课将探讨 **导数**。

## 本课剧情：用放大镜看变化

中心差分（central difference，CD）把导数变成可以计算的操作：在 x 左右各取一个点，用输出差除以间距 2h。h 取 1e-5 时，误差已在 1e-10 量级——比单侧差分精确两个数量级。本节以 f(x) = x² 和 f(x) = x²+2x 为主要探索示例，实现 `numeric_derivative`，最后对照 sin(x) 的解析导数 cos(x) 验证实现的正确性。

## 学习目标

1. **陈述极限定义**：写出 f'(x) = lim_{h→0} (f(x+h)-f(x))/h，解释为何计算机必须用有限 h 近似。
2. **推导 O(h²) 误差**：说明中心差分为何比前向差分精确两个数量级（泰勒展开消去一阶项）。
3. **实现 `numeric_derivative`**：完成 TODO stub，用中心差分在任意点求导数近似值。
4. **实验找最优 h**：通过 h-sweep 实验观察误差先降后升的 U 形曲线，确认最优步长约 1e-5～1e-7。

## 1. 用数值方法近似导数（中心差分）

`f'(x) ≈ (f(x+h) - f(x-h)) / (2h)`，h 取很小的数。

中心差分之所以优于前向差分（forward difference） `(f(x+h) - f(x)) / h`，在于截断误差（truncation error）阶数不同。前向差分用 x 和 x+h 两点连线，斜率偏向右侧，误差是 O(h)。中心差分关于 x 对称取点，泰勒展开（Taylor expansion）时一阶误差项正负抵消，只剩 h² 项，误差是 O(h²)。h=1e-5 时，前向差分误差约 1e-5，中心差分误差约 1e-10——差两个数量级。

h 并非越小越好：当 h 缩到 1e-12 以下，`f(x+h) - f(x-h)` 两个几乎相等的浮点数（floating-point number）相减，有效位数从 16 位降到 4 位左右（catastrophic cancellation），误差反而急剧上升。最优 h 在 1e-5 到 1e-7 之间，此处截断误差与舍入误差（rounding error）平衡，总误差最小。

### 极限定义（limit definition）

导数的严格定义来自极限：

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

这描述了当区间 h 趋向 0 时，割线（secant line）斜率趋近切线（tangent line）斜率的过程。

**但计算机无法取 h→0**：浮点数精度有限（约 1e-16），h 太小时分子 `f(x+h)-f(x-h)` 两个近似相等的数相减，有效位数急剧损失（灾难性消除，catastrophic cancellation）。

因此实践中用有限 h（通常 h≈1e-5）的**中心差分**来近似：

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}, \quad h \approx 10^{-5}$$

这将误差从 O(h)（前向差分）压缩到 O(h²)（中心差分），h=1e-5 时误差约 1e-10。

## 实验入口：用数值变化观察函数

下面的实验在 x=3.0 处计算 f(x)=x² 的数值导数，关注 `h` 缩小时计算值如何接近真实斜率（真值=2x=6）。

In [ ]:
import numpy as np
f = lambda x: x**2
h = 1e-5
x = 3.0
print('f(x)=x^2 在 x=3 的斜率 ≈', (f(x+h)-f(x-h))/(2*h), ' (真值 2x=6)')

In [ ]:
# 切线可视化：函数曲线 + 切线
import numpy as np
import matplotlib.pyplot as plt

x_arr = np.linspace(0, 5, 300)
f_arr = x_arr**2

x0 = 3.0
slope = 2 * x0  # f'(x0) = 2x0 = 6（解析值）
tangent = f(x0) + slope * (x_arr - x0)  # 切线方程

plt.figure(figsize=(6, 4))
plt.plot(x_arr, f_arr, label=r"$f(x)=x^2$", linewidth=2)
plt.plot(x_arr, tangent, "--", label=f"切线 @ x={x0}（斜率={slope}）", linewidth=1.5)
plt.scatter([x0], [x0**2], color="red", zorder=5, label=f"x={x0}")
plt.xlim(0, 5); plt.ylim(-5, 25)
plt.xlabel("x"); plt.ylabel("f(x)")
plt.title("切线斜率 = 导数值")
plt.legend(); plt.tight_layout(); plt.show()
print(f"x={x0} 处：解析斜率={slope}，数值斜率（若已实现）将在验证 cell 中对比")

## 动手观察：多点斜率一览

对 f(x)=x² 在 xs=[-2,-1,0,1,2] 五个点上批量计算数值斜率（向量化中心差分），与真值 2x 逐点比较，验证误差一致在 1e-3 量级（此处 h=1e-3）。

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('近似斜率 =', np.round(slopes, 3))
print('理论斜率 2x =', 2 * xs)


## 代码实验：遍历不同位置，看斜率如何变化

对 f(x)=x²+2x 在 [-3, 3] 上取 7 个点，逐点打印数值导数与解析导数（2x+2）的差值，验证中心差分在整个区间都准确。

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ 实现 `numeric_derivative(f, x, h=1e-5)`

**推理路线**：
1. 函数签名：接受 `f`（任意可调用）、`x`（float，求导点）、`h`（扰动步长，默认 1e-5）。
2. 计算 `f(x+h)` 和 `f(x-h)`——各向右/左扰动 h，共两次函数求值。
3. 两者相减后除以 `2*h`，得到中心差分近似，返回单个 float。

**参考输入输出**：f=sin, x=0, h=1e-5 → `(sin(1e-5) - sin(-1e-5))/(2e-5)` ≈ `(1e-5 - (-1e-5))/(2e-5)` = 1.0 = cos(0) ✓

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `numeric_derivative` 前明确三件事：
- 输入：`f`（可调用函数）、`x`（求导点）、`h`（扰动步长，默认 1e-5）
- 关键步骤：计算 `f(x+h)` 和 `f(x-h)`，相减后除以 `2h`
- 返回：单个浮点数，表示 f 在 x 处的斜率近似值

In [ ]:
def numeric_derivative(f, x, h=1e-5):
    # ✏️ TODO: 中心差分近似导数
    raise NotImplementedError("TODO: 实现中心差分近似 — 用 (f(x+h) - f(x-h)) / (2*h)")

In [ ]:
try:
    assert abs(numeric_derivative(lambda x: x**2, 3.0) - 6.0) < 1e-8, "x²在x=3处导数应≈6.0，误差应<1e-8"
    assert abs(numeric_derivative(np.sin, 0.0) - 1.0) < 1e-8, "sin在x=0处导数应≈1.0，误差应<1e-8"
    print("✅ 通过：你会数值求导了。")
except NotImplementedError:
    print("⚠️ 还没实现 numeric_derivative，请完成 TODO 后重新运行。")
except AssertionError as e:
    print(f"❌ 断言失败：{e}")

## 3. 记几个常见导数（背下来省事）

| f(x) | f'(x) |
|---|---|
| xⁿ | n·xⁿ⁻¹ |
| eˣ | eˣ |
| sin x | cos x |
| ln x | 1/x |

**🔗 Aurora 连接**：`numeric_derivative` 将在后续里程碑中集成到 `src/aurora/audio/grad_check.py`（尚未创建），用于将 MFCC、梅尔滤波器等模块的解析梯度与数值梯度逐元素比对，差异超过 1e-5 即判定求导实现有误。目前可在当前 notebook 中直接使用实现的函数进行验证。下一节（`L23_gradients.ipynb`）把单点斜率推广到多变量：沿每个参数维度重复一次 `numeric_derivative`，拼成梯度向量。

In [ ]:
# 参数实验：h 的最优区间（与 cell[17] 描述对应）
hs = [1e-1, 1e-2, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11, 1e-13, 1e-15]
print(f"{'h':>10}  {'|误差|':>12}")
print('-' * 26)
for h in hs:
    err = abs(numeric_derivative(np.sin, 0, h) - 1.0)  # 精确值 cos(0)=1
    print(f"{h:10.0e}  {err:12.2e}")
print("\n最优 h 约 1e-5～1e-7，误差约 1e-10；h 继续缩小后浮点舍入误差反而上升。")


## 参数实验：h 的最优区间

把 `h` 从 1e-1 逐步减小到 1e-15，每步打印 `abs(numeric_derivative(np.sin, 0, h) - 1.0)`，观察误差先减后增的 U 形曲线，找到最优步长约 1e-5~1e-7。

- **h 偏大（1e-1 到 1e-3）**：截断误差主导，误差约 h²/6·f'''(x)，随 h 缩小以平方速率下降。
- **h 最优（1e-5 到 1e-7）**：截断误差与浮点舍入误差平衡，总误差最小约 1e-10。
- **h 过小（1e-12 以下）**：`f(x+h) - f(x-h)` 两个近似相等的数相减，有效位数从 16 位骤降至 4 位以下，误差反而随 h 减小而上升。

## 本课收束

現在可以用 `numeric_derivative(f, x, h=1e-5)` 在任意点计算可调用函数的斜率，输出单个浮点数，误差量级 1e-10。这个函数在 Aurora 梯度检验中直接使用：将损失对权重的解析梯度与数值梯度逐元素比对，差异超过 1e-5 即报错。`h` 的最优范围在 1e-5 附近，过大则截断误差主导，过小则浮点舍入误差主导。

下一课：**L23**（梯度）把这个操作沿每个参数维度重复，得到多变量函数的梯度向量。

---

→ **下一课**　[L23 · 梯度](L23_gradients.ipynb)

> 下节课将学习 **梯度**：多元函数的"最陡上坡"方向，偏导与梯度向量的计算。